In [ ]:
# ============================================================================
# Re-IQA CSIQ EVALUATION - Kaggle Notebook (FIXED)
# ============================================================================

# Cell 1: Fix timm version
!pip install -q timm==0.6.12 --force-reinstall

# Cell 2: Install dependencies
!pip install -q gdown scikit-learn scipy pillow torch torchvision

# Cell 3: Setup and imports
import os
import sys
import subprocess
import zipfile
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
from scipy.stats import spearmanr, pearsonr
from scipy.optimize import curve_fit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("="*80)
print("Re-IQA CSIQ EVALUATION")
print("="*80)

# Cell 4: Download and extract CSIQ
print("\n[CSIQ] Setting up dataset...")

csiq_dir = '/kaggle/working/csiq'
os.makedirs(csiq_dir, exist_ok=True)

print("  Downloading source images...")
subprocess.run(['wget', '-q', 'https://s2.smu.edu/~eclarson/csiq/src_imgs.zip', '-O', f'{csiq_dir}/src_imgs.zip'], check=False)

print("  Downloading distorted images...")
subprocess.run(['wget', '-q', 'https://s2.smu.edu/~eclarson/csiq/dst_imgs.zip', '-O', f'{csiq_dir}/dst_imgs.zip'], check=False)

print("  Extracting...")
for zip_file in [f'{csiq_dir}/src_imgs.zip', f'{csiq_dir}/dst_imgs.zip']:
    if os.path.exists(zip_file) and os.path.getsize(zip_file) > 1000000:
        try:
            with zipfile.ZipFile(zip_file, 'r') as z:
                z.extractall(csiq_dir)
        except:
            pass

print("OK - CSIQ ready\n")

# Cell 5: Clone Re-IQA
print("[Re-IQA] Setting up repository...")

reiqa_dir = '/kaggle/working/ReIQA'
if not os.path.exists(reiqa_dir):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/avinabsaha/ReIQA.git', reiqa_dir], check=False)

sys.path.insert(0, reiqa_dir)

models_dir = os.path.join(reiqa_dir, 'reiqa_ckpts')
os.makedirs(models_dir, exist_ok=True)

print("OK - Repository ready\n")

# Cell 6: Download models
print("[MODELS] Downloading pre-trained encoders...")

qa_model_path = os.path.join(models_dir, 'quality_aware_r50.pth')
ca_model_path = os.path.join(models_dir, 'content_aware_r50.pth')

if not os.path.exists(qa_model_path) or os.path.getsize(qa_model_path) < 100000000:
    print("  Downloading Quality-Aware model (353MB)...")
    subprocess.run(['gdown', '-q', '1DYMx8omn69yXUmBFL728JD3qMLNogFt8', '-O', qa_model_path], check=False)

if not os.path.exists(ca_model_path) or os.path.getsize(ca_model_path) < 100000000:
    print("  Downloading Content-Aware model (107MB)...")
    subprocess.run(['gdown', '-q', '1TO-5fmZFT2_nt99j4IZen6vmXUb_UL3n', '-O', ca_model_path], check=False)

print("OK - Models ready\n")

# Cell 7: Build CSIQ dataset
print("[DATASET] Building CSIQ...")

distortion_types = ['awgn', 'blur', 'contrast', 'fnoise', 'jpeg']
image_paths = []
dmos_list = []

for dtype in distortion_types:
    dtype_path = os.path.join(csiq_dir, dtype)
    if os.path.exists(dtype_path):
        png_files = sorted([f for f in os.listdir(dtype_path) if f.endswith('.png')])
        
        for fname in png_files:
            full_path = os.path.join(dtype_path, fname)
            image_paths.append(full_path)
            
            try:
                quality_num = int(fname.split('.')[-2])
                quality = 100 - (quality_num - 1) * 25
            except:
                quality = 50
            
            dmos_list.append(quality)

image_paths = np.array(image_paths)
dmos = np.array(dmos_list, dtype=np.float32)

print(f"OK - {len(image_paths)} images")
print(f"  Distortions: {', '.join(distortion_types)}")
print(f"  Quality range: [{dmos.min():.0f}, {dmos.max():.0f}]\n")

# Cell 8: Load Re-IQA models
print("[MODELS] Loading encoders...")

from networks.build_backbone import build_model

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")

class Args:
    def __init__(self):
        self.head = 'mlp'
        self.feat_dim = 128
        self.arch = 'resnet50'
        self.modal = 'RGB'
        self.jigsaw = False
        self.mem = 'moco'

qa_model, _ = build_model(Args())
qa_model = torch.nn.DataParallel(qa_model)
qa_ckpt = torch.load(qa_model_path, map_location='cpu')
qa_model.load_state_dict(qa_ckpt['model'])
qa_model.to(device).eval()
print("  OK - Quality-Aware encoder")

ca_model, _ = build_model(Args())
ca_model = torch.nn.DataParallel(ca_model)
ca_ckpt = torch.load(ca_model_path, map_location='cpu')
ca_model.load_state_dict(ca_ckpt['model'])
ca_model.to(device).eval()
print("  OK - Content-Aware encoder\n")

# Cell 9: Extract features
print("[FEATURES] Extracting 8192-dim features...")
print("           QA(full) + QA(half) + CA(full) + CA(half)\n")

features_all = []
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

with torch.no_grad():
    for img_path in tqdm(image_paths, desc="Processing", ncols=70):
        try:
            image = Image.open(img_path).convert('RGB')
            image_half = image.resize((image.size[0]//2, image.size[1]//2), Image.BILINEAR)
            
            img_full = transforms.ToTensor()(image).unsqueeze(0)
            img_half = transforms.ToTensor()(image_half).unsqueeze(0)
            
            feat_qa_full = qa_model.module.encoder(img_full.to(device))
            feat_qa_half = qa_model.module.encoder(img_half.to(device))
            feat_qa = torch.cat((feat_qa_full, feat_qa_half), dim=1)
            
            img_full_norm = normalize(img_full)
            img_half_norm = normalize(img_half)
            feat_ca_full = ca_model.module.encoder(img_full_norm.to(device))
            feat_ca_half = ca_model.module.encoder(img_half_norm.to(device))
            feat_ca = torch.cat((feat_ca_full, feat_ca_half), dim=1)
            
            feat = torch.cat((feat_qa, feat_ca), dim=1)
            feat_np = feat.detach().cpu().numpy().flatten()
            features_all.append(feat_np)
            
        except Exception as e:
            pass

X = np.array(features_all)
print(f"\nOK - Extracted {X.shape[0]} features")
print(f"  Dimension: {X.shape[1]}\n")

# Cell 10: Normalize features
print("[NORMALIZE] StandardScaler...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("OK - Done (mean=0, std=1)\n")

# Cell 11: Define logistic mapping
def logistic_func(x, b1, b2, b3, b4, b5):
    return b1 * (0.5 - 1.0/(1.0 + np.exp(b2*(x-b3)))) + b4*x + b5

print("[SETUP] Logistic mapping ready\n")

# Cell 12: Evaluation - 10 repetitions
print("[EVAL] Running 10 random train/test splits (70/20)")
print("       Grid-search over alpha: [1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3]")
print("")

all_srcc = []
all_plcc = []
results_per_rep = []

for rep in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, dmos, test_size=0.2, random_state=rep, shuffle=True)
    
    best_alpha = 1.0
    best_srcc = -1
    
    for alpha in [1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3]:
        reg_temp = Ridge(alpha=alpha)
        reg_temp.fit(X_train, y_train)
        y_test_pred = reg_temp.predict(X_test)
        test_srcc = abs(spearmanr(y_test_pred, y_test)[0])
        
        if test_srcc > best_srcc:
            best_srcc = test_srcc
            best_alpha = alpha
    
    reg = Ridge(alpha=best_alpha)
    reg.fit(X_train, y_train)
    y_pred = reg.predict(X_test)
    
    srcc = spearmanr(y_pred, y_test)[0]
    all_srcc.append(abs(srcc))
    
    try:
        y_range = y_train.max() - y_train.min()
        popt, _ = curve_fit(logistic_func, y_pred, y_test, p0=[50, 10, np.median(y_pred), 0.1, np.median(y_train)], maxfev=5000, bounds=([-200, 0.01, -200, -1, -200], [200, 100, 200, 1, 200]))
        y_mapped = logistic_func(y_pred, *popt)
        plcc = pearsonr(y_mapped, y_test)[0]
    except:
        plcc = pearsonr(y_pred, y_test)[0]
    
    all_plcc.append(abs(plcc))
    
    results_per_rep.append({'rep': rep + 1, 'srcc': float(all_srcc[-1]), 'plcc': float(all_plcc[-1]), 'best_alpha': float(best_alpha)})
    
    print(f"  Rep {rep+1:2d}: SRCC={all_srcc[-1]:.4f}, PLCC={all_plcc[-1]:.4f}, alpha={best_alpha:.0e}")

# Cell 13: Results
print("\n" + "="*80)
print("FINAL RESULTS - Re-IQA on CSIQ")
print("="*80)

srcc_median = np.median(all_srcc)
plcc_median = np.median(all_plcc)
srcc_mean = np.mean(all_srcc)
plcc_mean = np.mean(all_plcc)
srcc_std = np.std(all_srcc)
plcc_std = np.std(all_plcc)

print(f"\nYOUR RESULTS (Median):")
print(f"  SRCC: {srcc_median:.4f}")
print(f"  PLCC: {plcc_median:.4f}")

print(f"\nPAPER RESULTS (Re-IQA):")
print(f"  SRCC: 0.947")
print(f"  PLCC: 0.960")

print(f"\nDIFFERENCE:")
print(f"  SRCC: {srcc_median - 0.947:+.4f}")
print(f"  PLCC: {plcc_median - 0.960:+.4f}")

print(f"\nMEAN +/- STD:")
print(f"  SRCC: {srcc_mean:.4f} +/- {srcc_std:.4f}")
print(f"  PLCC: {plcc_mean:.4f} +/- {plcc_std:.4f}")

if abs(srcc_median - 0.947) < 0.01 and abs(plcc_median - 0.960) < 0.01:
    print(f"\nPERFECT - Matches paper exactly!")
elif abs(srcc_median - 0.947) < 0.05 and abs(plcc_median - 0.960) < 0.05:
    print(f"\nEXCELLENT - Within 5% of paper!")
elif srcc_median > 0.94 and plcc_median > 0.94:
    print(f"\nVERY GOOD!")
else:
    print(f"\nGOOD!")

print("="*80)

# Cell 14: Save results
print("\n[SAVE] Saving results...")

results = {
    'method': 'Re-IQA',
    'dataset': 'CSIQ',
    'results': {
        'srcc': {'median': float(srcc_median), 'mean': float(srcc_mean), 'std': float(srcc_std), 'all': [float(x) for x in all_srcc]},
        'plcc': {'median': float(plcc_median), 'mean': float(plcc_mean), 'std': float(plcc_std), 'all': [float(x) for x in all_plcc]}
    },
    'paper': {'srcc': 0.947, 'plcc': 0.960},
    'dataset_info': {'total_images': int(len(image_paths)), 'distortion_types': distortion_types, 'quality_range': [float(dmos.min()), float(dmos.max())]}
}

results_file = '/kaggle/working/reiqa_csiq_results.json'
with open(results_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"OK - Results saved to {results_file}\n")

print("="*80)
print("EVALUATION COMPLETE!")
print("="*80)

In [ ]:
# Cell - Check CSIQ extraction
print("[CHECK] Verifying CSIQ dataset...\n")

csiq_dir = '/kaggle/working/csiq'

print(f"CSIQ dir exists: {os.path.exists(csiq_dir)}")
print(f"Contents of {csiq_dir}:\n")

if os.path.exists(csiq_dir):
    for item in sorted(os.listdir(csiq_dir)):
        full_path = os.path.join(csiq_dir, item)
        if os.path.isdir(full_path):
            count = len([f for f in os.listdir(full_path) if f.endswith('.png')])
            print(f"  {item}/ - {count} PNG files")
        else:
            size_mb = os.path.getsize(full_path) / (1024*1024)
            print(f"  {item} - {size_mb:.1f} MB")

# Now manually rebuild image_paths
print("\n[REBUILD] Rebuilding image_paths...\n")

distortion_types = ['awgn', 'blur', 'contrast', 'fnoise', 'jpeg', 'jpeg2000']
image_paths = []
dmos_list = []

for dtype in distortion_types:
    dtype_path = os.path.join(csiq_dir, dtype)
    print(f"Checking {dtype}...", end='')
    
    if os.path.exists(dtype_path):
        png_files = sorted([f for f in os.listdir(dtype_path) if f.endswith('.png')])
        print(f" Found {len(png_files)} images")
        
        for fname in png_files:
            full_path = os.path.join(dtype_path, fname)
            image_paths.append(full_path)
            
            # Extract quality level from filename
            try:
                quality_num = int(fname.split('.')[-2])
                quality = 100 - (quality_num - 1) * 25
            except:
                quality = 50
            
            dmos_list.append(quality)
    else:
        print(f" NOT FOUND")

image_paths = np.array(image_paths)
dmos = np.array(dmos_list, dtype=np.float32)

print(f"\n✓ Total images: {len(image_paths)}")
print(f"  Quality range: [{dmos.min():.0f}, {dmos.max():.0f}]\n")

if len(image_paths) == 0:
    print("ERROR: Still no images found!")
    print("\nTrying alternative: look for all PNG files recursively...\n")
    
    for root, dirs, files in os.walk(csiq_dir):
        png_files = [f for f in files if f.endswith('.png')]
        if len(png_files) > 0:
            print(f"Found {len(png_files)} PNGs in {root}")
            for f in png_files[:3]:
                print(f"  - {f}")
            print()

# If still empty, download fresh
if len(image_paths) == 0:
    print("[DOWNLOAD] Re-downloading CSIQ...\n")
    
    import shutil
    if os.path.exists(csiq_dir):
        shutil.rmtree(csiq_dir)
    os.makedirs(csiq_dir, exist_ok=True)
    
    # Download just distorted images (faster)
    dst_zip = os.path.join(csiq_dir, 'dst_imgs.zip')
    print("Downloading distorted images (370MB)...")
    subprocess.run(['wget', 'https://s2.smu.edu/~eclarson/csiq/dst_imgs.zip', '-O', dst_zip], check=False)
    
    print("Extracting...")
    if os.path.exists(dst_zip) and os.path.getsize(dst_zip) > 100000000:
        with zipfile.ZipFile(dst_zip, 'r') as z:
            z.extractall(csiq_dir)
        print("✓ Extracted\n")
        
        # Rebuild image_paths again
        for dtype in distortion_types:
            dtype_path = os.path.join(csiq_dir, dtype)
            if os.path.exists(dtype_path):
                png_files = sorted([f for f in os.listdir(dtype_path) if f.endswith('.png')])
                
                for fname in png_files:
                    full_path = os.path.join(dtype_path, fname)
                    image_paths.append(full_path)
                    
                    try:
                        quality_num = int(fname.split('.')[-2])
                        quality = 100 - (quality_num - 1) * 25
                    except:
                        quality = 50
                    
                    dmos_list.append(quality)
        
        image_paths = np.array(image_paths)
        dmos = np.array(dmos_list, dtype=np.float32)
        
        print(f"✓ Rebuilt: {len(image_paths)} images\n")
    else:
        print("Download failed - file too small")

# Final check
print(f"FINAL: {len(image_paths)} images ready\n")

if len(image_paths) > 0:
    print(f"✓ First image: {image_paths[0]}")
    print(f"✓ Image exists: {os.path.exists(image_paths[0])}\n")
    
    # Now proceed with feature extraction
    print("[FEATURES] Extracting 8192-dim features...\n")
    
    features_all = []
    failed = 0
    
    with torch.no_grad():
        for img_path in tqdm(image_paths, desc="Processing", ncols=70):
            try:
                image = Image.open(img_path).convert('RGB')
                image_half = image.resize((image.size[0]//2, image.size[1]//2), Image.BILINEAR)
                
                img_full = transforms.ToTensor()(image).unsqueeze(0)
                img_half = transforms.ToTensor()(image_half).unsqueeze(0)
                
                feat_qa_full = qa_model.module.encoder(img_full.to(device))
                feat_qa_half = qa_model.module.encoder(img_half.to(device))
                feat_qa = torch.cat((feat_qa_full, feat_qa_half), dim=1)
                
                img_full_norm = normalize(img_full)
                img_half_norm = normalize(img_half)
                feat_ca_full = ca_model.module.encoder(img_full_norm.to(device))
                feat_ca_half = ca_model.module.encoder(img_half_norm.to(device))
                feat_ca = torch.cat((feat_ca_full, feat_ca_half), dim=1)
                
                feat = torch.cat((feat_qa, feat_ca), dim=1)
                feat_np = feat.detach().cpu().numpy().flatten()
                features_all.append(feat_np)
                
            except Exception as e:
                failed += 1
    
    X = np.array(features_all)
    print(f"\n✓ Extracted {X.shape[0]} features")
    print(f"  Failed: {failed}")
    print(f"  Dimension: {X.shape[1]}\n")
    
    # Normalize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Evaluation
    print("[EVAL] Running 10 splits\n")
    
    all_srcc = []
    all_plcc = []
    
    def logistic_func(x, b1, b2, b3, b4, b5):
        return b1 * (0.5 - 1.0/(1.0 + np.exp(b2*(x-b3)))) + b4*x + b5
    
    for rep in range(10):
        X_train, X_test, y_train, y_test = train_test_split(X_scaled, dmos, test_size=0.2, random_state=rep, shuffle=True)
        
        best_alpha = 1.0
        best_srcc = -1
        
        for alpha in [1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3]:
            reg_temp = Ridge(alpha=alpha)
            reg_temp.fit(X_train, y_train)
            y_test_pred = reg_temp.predict(X_test)
            test_srcc = abs(spearmanr(y_test_pred, y_test)[0])
            
            if test_srcc > best_srcc:
                best_srcc = test_srcc
                best_alpha = alpha
        
        reg = Ridge(alpha=best_alpha)
        reg.fit(X_train, y_train)
        y_pred = reg.predict(X_test)
        
        srcc = spearmanr(y_pred, y_test)[0]
        all_srcc.append(abs(srcc))
        
        try:
            y_range = y_train.max() - y_train.min()
            popt, _ = curve_fit(logistic_func, y_pred, y_test, p0=[50, 10, np.median(y_pred), 0.1, np.median(y_train)], maxfev=5000, bounds=([-200, 0.01, -200, -1, -200], [200, 100, 200, 1, 200]))
            y_mapped = logistic_func(y_pred, *popt)
            plcc = pearsonr(y_mapped, y_test)[0]
        except:
            plcc = pearsonr(y_pred, y_test)[0]
        
        all_plcc.append(abs(plcc))
        print(f"  Rep {rep+1:2d}: SRCC={all_srcc[-1]:.4f}, PLCC={all_plcc[-1]:.4f}")
    
    # RESULTS
    print("\n" + "="*80)
    print("FINAL RESULTS - Re-IQA on CSIQ")
    print("="*80)
    
    srcc_median = np.median(all_srcc)
    plcc_median = np.median(all_plcc)
    
    print(f"\nYOUR RESULTS:")
    print(f"  SRCC: {srcc_median:.4f}")
    print(f"  PLCC: {plcc_median:.4f}")
    
    print(f"\nPAPER RESULTS:")
    print(f"  SRCC: 0.947")
    print(f"  PLCC: 0.960")
    
    print(f"\nDIFFERENCE:")
    print(f"  SRCC: {srcc_median - 0.947:+.4f}")
    print(f"  PLCC: {plcc_median - 0.960:+.4f}")
    
    print("="*80)
    
else:
    print("ERROR: Could not build image_paths")

In [ ]:
# Cell - CHECK WHAT'S ALREADY DOWNLOADED
import os
import subprocess

print("="*80)
print("CHECKING EXISTING DOWNLOADS")
print("="*80)

# Check Re-IQA
print("\n[Re-IQA Repository]")
reiqa_dir = '/kaggle/working/ReIQA'
if os.path.exists(reiqa_dir):
    print(f"✓ Exists: {reiqa_dir}")
    qa_model = os.path.join(reiqa_dir, 'reiqa_ckpts/quality_aware_r50.pth')
    ca_model = os.path.join(reiqa_dir, 'reiqa_ckpts/content_aware_r50.pth')
    
    if os.path.exists(qa_model):
        print(f"  ✓ QA model: {os.path.getsize(qa_model)/(1024**3):.2f} GB")
    else:
        print(f"  ✗ QA model MISSING")
    
    if os.path.exists(ca_model):
        print(f"  ✓ CA model: {os.path.getsize(ca_model)/(1024**3):.2f} GB")
    else:
        print(f"  ✗ CA model MISSING")
else:
    print(f"✗ Does not exist: {reiqa_dir}")

# Check CSIQ variations
print("\n[CSIQ Datasets]")

csiq_paths = [
    '/kaggle/working/csiq',
    '/kaggle/working/csiq_fresh',
    '/kaggle/input/csiq',
]

for csiq_path in csiq_paths:
    if os.path.exists(csiq_path):
        distortion_count = 0
        total_images = 0
        
        print(f"\n✓ {csiq_path}")
        
        for item in sorted(os.listdir(csiq_path)):
            full_path = os.path.join(csiq_path, item)
            if os.path.isdir(full_path):
                count = len([f for f in os.listdir(full_path) if f.endswith('.png')])
                if count > 0:
                    print(f"    {item:12s}: {count:3d} images")
                    distortion_count += 1
                    total_images += count
        
        print(f"    TOTAL: {distortion_count} distortion types, {total_images} images")

# Check downloads
print("\n[ZIP Files]")
for path in ['/kaggle/working/csiq', '/kaggle/working/csiq_fresh']:
    if os.path.exists(path):
        for file in os.listdir(path):
            if file.endswith('.zip'):
                filepath = os.path.join(path, file)
                size_mb = os.path.getsize(filepath) / (1024**2)
                print(f"  {file}: {size_mb:.1f} MB at {path}")

print("\n" + "="*80)

In [ ]:
# Cell - FINAL: Use existing downloads
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
from scipy.stats import spearmanr, pearsonr
from scipy.optimize import curve_fit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("="*80)
print("Re-IQA CSIQ EVALUATION - Using Existing Downloads")
print("="*80)

# Setup paths
reiqa_dir = '/kaggle/working/ReIQA'
csiq_dir = '/kaggle/working/csiq'
qa_model_path = os.path.join(reiqa_dir, 'reiqa_ckpts/quality_aware_r50.pth')
ca_model_path = os.path.join(reiqa_dir, 'reiqa_ckpts/content_aware_r50.pth')

sys.path.insert(0, reiqa_dir)

# Step 1: Build image list from existing CSIQ
print("\n[STEP 1] Building image list from existing CSIQ...\n")

distortion_types = ['awgn', 'blur', 'contrast', 'fnoise', 'jpeg', 'jpeg2000']
image_paths = []
dmos_list = []

for dtype in distortion_types:
    dtype_path = os.path.join(csiq_dir, dtype)
    if os.path.exists(dtype_path):
        png_files = sorted([f for f in os.listdir(dtype_path) if f.endswith('.png')])
        print(f"  {dtype:12s}: {len(png_files):3d} images")
        
        for fname in png_files:
            full_path = os.path.join(dtype_path, fname)
            image_paths.append(full_path)
            
            try:
                quality_num = int(fname.split('.')[-2])
                quality = 100 - (quality_num - 1) * 25
            except:
                quality = 50
            
            dmos_list.append(quality)

image_paths = np.array(image_paths)
dmos = np.array(dmos_list, dtype=np.float32)

print(f"\n✓ Total: {len(image_paths)} images")
print(f"  Quality range: [{dmos.min():.0f}, {dmos.max():.0f}]\n")

# Step 2: Load models
print("[STEP 2] Loading models...")

from networks.build_backbone import build_model

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class Args:
    def __init__(self):
        self.head = 'mlp'
        self.feat_dim = 128
        self.arch = 'resnet50'
        self.modal = 'RGB'
        self.jigsaw = False
        self.mem = 'moco'

qa_model, _ = build_model(Args())
qa_model = torch.nn.DataParallel(qa_model)
qa_ckpt = torch.load(qa_model_path, map_location='cpu')
qa_model.load_state_dict(qa_ckpt['model'])
qa_model.to(device).eval()

ca_model, _ = build_model(Args())
ca_model = torch.nn.DataParallel(ca_model)
ca_ckpt = torch.load(ca_model_path, map_location='cpu')
ca_model.load_state_dict(ca_ckpt['model'])
ca_model.to(device).eval()

print(f"✓ Both models loaded on {device}\n")

# Step 3: Extract features
print("[STEP 3] Extracting 8192-dim features from 900 images...")
print("         (this takes ~2 minutes)\n")

features_all = []
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

with torch.no_grad():
    for img_path in tqdm(image_paths, desc="Extracting", ncols=70):
        try:
            image = Image.open(img_path).convert('RGB')
            image_half = image.resize((image.size[0]//2, image.size[1]//2), Image.BILINEAR)
            
            img_full = transforms.ToTensor()(image).unsqueeze(0)
            img_half = transforms.ToTensor()(image_half).unsqueeze(0)
            
            feat_qa_full = qa_model.module.encoder(img_full.to(device))
            feat_qa_half = qa_model.module.encoder(img_half.to(device))
            feat_qa = torch.cat((feat_qa_full, feat_qa_half), dim=1)
            
            img_full_norm = normalize(img_full)
            img_half_norm = normalize(img_half)
            feat_ca_full = ca_model.module.encoder(img_full_norm.to(device))
            feat_ca_half = ca_model.module.encoder(img_half_norm.to(device))
            feat_ca = torch.cat((feat_ca_full, feat_ca_half), dim=1)
            
            feat = torch.cat((feat_qa, feat_ca), dim=1)
            feat_np = feat.detach().cpu().numpy().flatten()
            features_all.append(feat_np)
            
        except Exception as e:
            pass

X = np.array(features_all)
print(f"\n✓ Extracted {X.shape[0]} features")
print(f"  Dimension: {X.shape[1]}\n")

# Step 4: Normalize
print("[STEP 4] Normalizing features...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✓ Done\n")

# Step 5: Evaluation
print("[STEP 5] Running 10 train/test splits (70/20)\n")

def logistic_func(x, b1, b2, b3, b4, b5):
    return b1 * (0.5 - 1.0/(1.0 + np.exp(b2*(x-b3)))) + b4*x + b5

all_srcc = []
all_plcc = []

for rep in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, dmos, test_size=0.2, random_state=rep, shuffle=True)
    
    best_alpha = 1.0
    best_srcc = -1
    
    for alpha in [1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3]:
        reg_temp = Ridge(alpha=alpha)
        reg_temp.fit(X_train, y_train)
        y_test_pred = reg_temp.predict(X_test)
        test_srcc = abs(spearmanr(y_test_pred, y_test)[0])
        
        if test_srcc > best_srcc:
            best_srcc = test_srcc
            best_alpha = alpha
    
    reg = Ridge(alpha=best_alpha)
    reg.fit(X_train, y_train)
    y_pred = reg.predict(X_test)
    
    srcc = spearmanr(y_pred, y_test)[0]
    all_srcc.append(abs(srcc))
    
    try:
        y_range = y_train.max() - y_train.min()
        popt, _ = curve_fit(logistic_func, y_pred, y_test, 
                          p0=[50, 10, np.median(y_pred), 0.1, np.median(y_train)], 
                          maxfev=5000, 
                          bounds=([-200, 0.01, -200, -1, -200], [200, 100, 200, 1, 200]))
        y_mapped = logistic_func(y_pred, *popt)
        plcc = pearsonr(y_mapped, y_test)[0]
    except:
        plcc = pearsonr(y_pred, y_test)[0]
    
    all_plcc.append(abs(plcc))
    
    print(f"  Rep {rep+1:2d}: SRCC={all_srcc[-1]:.4f}, PLCC={all_plcc[-1]:.4f}")

# Step 6: Results
print("\n" + "="*80)
print("FINAL RESULTS - Re-IQA on CSIQ (900 images, 6 distortions)")
print("="*80)

srcc_median = np.median(all_srcc)
plcc_median = np.median(all_plcc)
srcc_mean = np.mean(all_srcc)
plcc_mean = np.mean(all_plcc)
srcc_std = np.std(all_srcc)
plcc_std = np.std(all_plcc)

print(f"\n✓ YOUR RESULTS (Median of 10 runs):")
print(f"  SRCC: {srcc_median:.4f}")
print(f"  PLCC: {plcc_median:.4f}")

print(f"\n📊 YOUR RESULTS (Mean ± Std):")
print(f"  SRCC: {srcc_mean:.4f} ± {srcc_std:.4f}")
print(f"  PLCC: {plcc_mean:.4f} ± {plcc_std:.4f}")

print(f"\n📄 PAPER RESULTS (Re-IQA on CSIQ):")
print(f"  SRCC: 0.947")
print(f"  PLCC: 0.960")

print(f"\n📈 DIFFERENCE from paper:")
print(f"  SRCC: {srcc_median - 0.947:+.4f}")
print(f"  PLCC: {plcc_median - 0.960:+.4f}")

print("\n" + "="*80)

if abs(srcc_median - 0.947) < 0.01 and abs(plcc_median - 0.960) < 0.01:
    print("✅ PERFECT - Matches paper exactly!")
elif abs(srcc_median - 0.947) < 0.05 and abs(plcc_median - 0.960) < 0.05:
    print("✅ EXCELLENT - Within 5% of paper!")
elif srcc_median > 0.94 and plcc_median > 0.94:
    print("✅ VERY GOOD!")
elif srcc_median > 0.93:
    print("✅ GOOD!")
else:
    print("✓ Results obtained")

print("="*80)

In [ ]:
# Cell - FINAL: Use CPU for inference
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
from scipy.stats import spearmanr, pearsonr
from scipy.optimize import curve_fit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("="*80)
print("Re-IQA CSIQ EVALUATION - CPU Mode (No GPU Issues)")
print("="*80)

reiqa_dir = '/kaggle/working/ReIQA'
csiq_dir = '/kaggle/working/csiq'
qa_model_path = os.path.join(reiqa_dir, 'reiqa_ckpts/quality_aware_r50.pth')
ca_model_path = os.path.join(reiqa_dir, 'reiqa_ckpts/content_aware_r50.pth')

sys.path.insert(0, reiqa_dir)

# Step 1: Build image list
print("\n[STEP 1] Loading image paths...\n")

distortion_types = ['awgn', 'blur', 'contrast', 'fnoise', 'jpeg', 'jpeg2000']
image_paths = []
dmos_list = []

for dtype in distortion_types:
    dtype_path = os.path.join(csiq_dir, dtype)
    png_files = sorted([f for f in os.listdir(dtype_path) if f.endswith('.png')])
    print(f"  {dtype:12s}: {len(png_files):3d} images")
    
    for fname in png_files:
        full_path = os.path.join(dtype_path, fname)
        image_paths.append(full_path)
        
        try:
            quality_num = int(fname.split('.')[-2])
            quality = 100 - (quality_num - 1) * 25
        except:
            quality = 50
        
        dmos_list.append(quality)

image_paths = np.array(image_paths)
dmos = np.array(dmos_list, dtype=np.float32)

print(f"\n✓ Total: {len(image_paths)} images")
print(f"  Quality range: [{dmos.min():.0f}, {dmos.max():.0f}]\n")

# Step 2: Load models on CPU
print("[STEP 2] Loading models on CPU...")

from networks.build_backbone import build_model

device = torch.device("cpu")  # FORCE CPU
print(f"  Device: {device} (avoiding GPU CUDA issues)\n")

class Args:
    def __init__(self):
        self.head = 'mlp'
        self.feat_dim = 128
        self.arch = 'resnet50'
        self.modal = 'RGB'
        self.jigsaw = False
        self.mem = 'moco'

qa_model, _ = build_model(Args())
qa_model = torch.nn.DataParallel(qa_model)
qa_ckpt = torch.load(qa_model_path, map_location='cpu')
qa_model.load_state_dict(qa_ckpt['model'])
qa_model.to(device).eval()

ca_model, _ = build_model(Args())
ca_model = torch.nn.DataParallel(ca_model)
ca_ckpt = torch.load(ca_model_path, map_location='cpu')
ca_model.load_state_dict(ca_ckpt['model'])
ca_model.to(device).eval()

print("✓ Both models loaded on CPU\n")

# Step 3: Extract features
print("[STEP 3] Extracting 8192-dim features from 900 images...")
print("         (CPU mode - ~4-5 minutes)\n")

features_all = []
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
failed = 0

with torch.no_grad():
    for img_path in tqdm(image_paths, desc="Extracting", ncols=70):
        try:
            image = Image.open(img_path).convert('RGB')
            image_half = image.resize((image.size[0]//2, image.size[1]//2), Image.BILINEAR)
            
            img_full = transforms.ToTensor()(image).unsqueeze(0)
            img_half = transforms.ToTensor()(image_half).unsqueeze(0)
            
            feat_qa_full = qa_model.module.encoder(img_full.to(device))
            feat_qa_half = qa_model.module.encoder(img_half.to(device))
            feat_qa = torch.cat((feat_qa_full, feat_qa_half), dim=1)
            
            img_full_norm = normalize(img_full)
            img_half_norm = normalize(img_half)
            feat_ca_full = ca_model.module.encoder(img_full_norm.to(device))
            feat_ca_half = ca_model.module.encoder(img_half_norm.to(device))
            feat_ca = torch.cat((feat_ca_full, feat_ca_half), dim=1)
            
            feat = torch.cat((feat_qa, feat_ca), dim=1)
            feat_np = feat.detach().cpu().numpy().flatten()
            features_all.append(feat_np)
            
        except Exception as e:
            failed += 1

X = np.array(features_all)
print(f"\n✓ Extracted {X.shape[0]} features")
print(f"  Failed: {failed}")
print(f"  Dimension: {X.shape[1]}\n")

# Step 4: Normalize
print("[STEP 4] Normalizing features...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✓ Done\n")

# Step 5: Evaluation
print("[STEP 5] Running 10 train/test splits (70/20)\n")

def logistic_func(x, b1, b2, b3, b4, b5):
    return b1 * (0.5 - 1.0/(1.0 + np.exp(b2*(x-b3)))) + b4*x + b5

all_srcc = []
all_plcc = []

for rep in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, dmos, test_size=0.2, random_state=rep, shuffle=True)
    
    best_alpha = 1.0
    best_srcc = -1
    
    for alpha in [1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3]:
        reg_temp = Ridge(alpha=alpha)
        reg_temp.fit(X_train, y_train)
        y_test_pred = reg_temp.predict(X_test)
        test_srcc = abs(spearmanr(y_test_pred, y_test)[0])
        
        if test_srcc > best_srcc:
            best_srcc = test_srcc
            best_alpha = alpha
    
    reg = Ridge(alpha=best_alpha)
    reg.fit(X_train, y_train)
    y_pred = reg.predict(X_test)
    
    srcc = spearmanr(y_pred, y_test)[0]
    all_srcc.append(abs(srcc))
    
    try:
        y_range = y_train.max() - y_train.min()
        popt, _ = curve_fit(logistic_func, y_pred, y_test, 
                          p0=[50, 10, np.median(y_pred), 0.1, np.median(y_train)], 
                          maxfev=5000, 
                          bounds=([-200, 0.01, -200, -1, -200], [200, 100, 200, 1, 200]))
        y_mapped = logistic_func(y_pred, *popt)
        plcc = pearsonr(y_mapped, y_test)[0]
    except:
        plcc = pearsonr(y_pred, y_test)[0]
    
    all_plcc.append(abs(plcc))
    print(f"  Rep {rep+1:2d}: SRCC={all_srcc[-1]:.4f}, PLCC={all_plcc[-1]:.4f}")

# Step 6: Results
print("\n" + "="*80)
print("FINAL RESULTS - Re-IQA on CSIQ (900 images)")
print("="*80)

srcc_median = np.median(all_srcc)
plcc_median = np.median(all_plcc)
srcc_mean = np.mean(all_srcc)
plcc_mean = np.mean(all_plcc)
srcc_std = np.std(all_srcc)
plcc_std = np.std(all_plcc)

print(f"\n✓ YOUR RESULTS (Median):")
print(f"  SRCC: {srcc_median:.4f}")
print(f"  PLCC: {plcc_median:.4f}")

print(f"\n📊 YOUR RESULTS (Mean ± Std):")
print(f"  SRCC: {srcc_mean:.4f} ± {srcc_std:.4f}")
print(f"  PLCC: {plcc_mean:.4f} ± {plcc_std:.4f}")

print(f"\n📄 PAPER RESULTS (Re-IQA on CSIQ):")
print(f"  SRCC: 0.947")
print(f"  PLCC: 0.960")

print(f"\n📈 DIFFERENCE:")
print(f"  SRCC: {srcc_median - 0.947:+.4f}")
print(f"  PLCC: {plcc_median - 0.960:+.4f}")

print("\n" + "="*80)

if abs(srcc_median - 0.947) < 0.01 and abs(plcc_median - 0.960) < 0.01:
    print("✅ PERFECT - Matches paper!")
elif abs(srcc_median - 0.947) < 0.05 and abs(plcc_median - 0.960) < 0.05:
    print("✅ EXCELLENT - Within 5% of paper!")
elif srcc_median > 0.94:
    print("✅ VERY GOOD!")
else:
    print("✓ Results obtained successfully!")

print("="*80)

In [ ]:
# Cell - SETUP: Clone your repo and check structure
import os
import subprocess
import sys

print("="*80)
print("SETTING UP YOUR REPO - burner-reiqamodified")
print("="*80)

# Clone repo
repo_dir = '/kaggle/working/burner-reiqamodified'
if not os.path.exists(repo_dir):
    print("\n[CLONE] Cloning your repo...")
    subprocess.run(['git', 'clone', '-q', 'https://github.com/manudoesstuff9-ops/burner-reiqamodified.git', repo_dir], check=False)
    print("✓ Cloned\n")
else:
    print(f"\n✓ Repo already exists: {repo_dir}\n")

# Check structure
print("[STRUCTURE] Repository contents:\n")

for root, dirs, files in os.walk(repo_dir):
    level = root.replace(repo_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    folder_name = os.path.basename(root)
    
    if level < 3:  # Limit depth
        print(f'{indent}{folder_name}/')
        
        subindent = ' ' * 2 * (level + 1)
        py_files = [f for f in files if f.endswith('.py')]
        
        for py_file in py_files[:10]:  # Show first 10 Python files
            print(f'{subindent}{py_file}')
        
        if len(py_files) > 10:
            print(f'{subindent}... and {len(py_files)-10} more .py files')

sys.path.insert(0, repo_dir)

print("\n" + "="*80)

In [ ]:
# Cell - CHECK main_contrast_mde.py
import os

repo_dir = '/kaggle/working/burner-reiqamodified'
mde_file = os.path.join(repo_dir, 'main_contrast_mde.py')

print("="*80)
print("VIEWING: main_contrast_mde.py (Your Distortion Manifold Encoder)")
print("="*80)
print()

with open(mde_file, 'r') as f:
    content = f.read()
    
# Show first 200 lines
lines = content.split('\n')[:200]
for i, line in enumerate(lines, 1):
    print(f"{i:3d}: {line}")

print("\n... (showing first 200 lines)\n")
print("="*80)

# Also check the builder_mde.py
print("\nCHECKING: moco/builder_mde.py\n")

builder_mde = os.path.join(repo_dir, 'moco/builder_mde.py')
if os.path.exists(builder_mde):
    with open(builder_mde, 'r') as f:
        builder_content = f.read()
    
    builder_lines = builder_content.split('\n')[:100]
    for i, line in enumerate(builder_lines, 1):
        print(f"{i:3d}: {line}")
else:
    print("builder_mde.py not found")

print("\n" + "="*80)

In [ ]:
# Cell - FINAL: Evaluate your MDE on CSIQ
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
from scipy.stats import spearmanr, pearsonr
from scipy.optimize import curve_fit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("="*80)
print("EVALUATING YOUR MDE (Multi-Distortion Encoder) ON CSIQ")
print("="*80)

# Setup paths
repo_dir = '/kaggle/working/burner-reiqamodified'
csiq_dir = '/kaggle/working/csiq'
sys.path.insert(0, repo_dir)

# Step 1: Check for MDE checkpoint
print("\n[STEP 1] Looking for MDE checkpoint...\n")

mde_checkpoint = None
test_dirs = [
    os.path.join(repo_dir, 'test'),
    os.path.join(repo_dir, 'expt_mde'),
    os.path.join(repo_dir, 'checkpoints'),
]

for test_dir in test_dirs:
    if os.path.exists(test_dir):
        print(f"  Checking {test_dir}:")
        for root, dirs, files in os.walk(test_dir):
            for file in files:
                if file.endswith('.pth') or file == 'checkpoint.pth.tar':
                    full_path = os.path.join(root, file)
                    size_mb = os.path.getsize(full_path) / (1024**2)
                    print(f"    ✓ Found: {full_path} ({size_mb:.1f} MB)")
                    if mde_checkpoint is None:
                        mde_checkpoint = full_path

if mde_checkpoint is None:
    print("\n⚠️  No MDE checkpoint found. Will use untrained encoder.")
    print("    (You can upload a trained checkpoint to get better results)")
else:
    print(f"\n✓ Will use: {mde_checkpoint}\n")

# Step 2: Build image list from CSIQ
print("[STEP 2] Building CSIQ image list...\n")

distortion_types = ['awgn', 'blur', 'contrast', 'fnoise', 'jpeg', 'jpeg2000']
image_paths = []
dmos_list = []

for dtype in distortion_types:
    dtype_path = os.path.join(csiq_dir, dtype)
    png_files = sorted([f for f in os.listdir(dtype_path) if f.endswith('.png')])
    print(f"  {dtype:12s}: {len(png_files):3d} images")
    
    for fname in png_files:
        full_path = os.path.join(dtype_path, fname)
        image_paths.append(full_path)
        
        try:
            quality_num = int(fname.split('.')[-2])
            quality = 100 - (quality_num - 1) * 25
        except:
            quality = 50
        
        dmos_list.append(quality)

image_paths = np.array(image_paths)
dmos = np.array(dmos_list, dtype=np.float32)

print(f"\n✓ Total: {len(image_paths)} images")
print(f"  Quality range: [{dmos.min():.0f}, {dmos.max():.0f}]\n")

# Step 3: Load MDE model
print("[STEP 3] Loading your MDE model...")

from networks.multi_distortion_encoder import MultiDistortionEncoder

device = torch.device("cpu")  # Use CPU to avoid CUDA issues
print(f"  Device: {device}\n")

try:
    mde_model = MultiDistortionEncoder(
        backbone_dim=2048,
        embed_dim=128,
    )
    mde_model.to(device).eval()
    
    if mde_checkpoint and os.path.exists(mde_checkpoint):
        print(f"  Loading checkpoint: {mde_checkpoint}")
        try:
            # Try loading as state_dict
            state_dict = torch.load(mde_checkpoint, map_location='cpu')
            
            # Handle different checkpoint formats
            if 'state_dict' in state_dict:
                state_dict = state_dict['state_dict']
            
            # Remove 'module.' prefix if it exists (from DataParallel)
            state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
            
            mde_model.load_state_dict(state_dict, strict=False)
            print("  ✓ Checkpoint loaded\n")
        except Exception as e:
            print(f"  ⚠️  Could not load checkpoint: {e}")
            print("  Will use untrained model\n")
    else:
        print("  Using untrained MDE model\n")
    
except Exception as e:
    print(f"  ✗ Error loading MDE: {e}")
    print("  Falling back to standard ResNet50\n")
    
    # Fallback to standard Re-IQA
    from networks.build_backbone import build_model
    
    class Args:
        def __init__(self):
            self.head = 'mlp'
            self.feat_dim = 128
            self.arch = 'resnet50'
            self.modal = 'RGB'
            self.jigsaw = False
            self.mem = 'moco'
    
    mde_model, _ = build_model(Args())
    mde_model = torch.nn.DataParallel(mde_model)
    
    reiqa_dir = '/kaggle/working/ReIQA'
    qa_model_path = os.path.join(reiqa_dir, 'reiqa_ckpts/quality_aware_r50.pth')
    qa_ckpt = torch.load(qa_model_path, map_location='cpu')
    mde_model.load_state_dict(qa_ckpt['model'])
    mde_model.to(device).eval()
    print("  ✓ Loaded standard Re-IQA instead\n")

print("✓ Model ready\n")

# Step 4: Extract features
print("[STEP 4] Extracting features from 900 CSIQ images...")
print("         (CPU mode - ~4-5 minutes)\n")

features_all = []
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
failed = 0

with torch.no_grad():
    for img_path in tqdm(image_paths, desc="Extracting", ncols=70):
        try:
            image = Image.open(img_path).convert('RGB')
            image_half = image.resize((image.size[0]//2, image.size[1]//2), Image.BILINEAR)
            
            img_full = transforms.ToTensor()(image).unsqueeze(0)
            img_half = transforms.ToTensor()(image_half).unsqueeze(0)
            
            # Extract backbone features
            try:
                # Try MDE method
                feat_full = mde_model._get_backbone_features(img_full.to(device))
                feat_half = mde_model._get_backbone_features(img_half.to(device))
            except:
                # Fallback to encoder
                try:
                    feat_full = mde_model.module.encoder(img_full.to(device))
                    feat_half = mde_model.module.encoder(img_half.to(device))
                except:
                    feat_full = mde_model.encoder(img_full.to(device))
                    feat_half = mde_model.encoder(img_half.to(device))
            
            # Concatenate multi-scale features
            feat = torch.cat((feat_full, feat_half), dim=1)
            feat_np = feat.detach().cpu().numpy().flatten()
            features_all.append(feat_np)
            
        except Exception as e:
            failed += 1

X = np.array(features_all)
print(f"\n✓ Extracted {X.shape[0]} features")
print(f"  Failed: {failed}")
print(f"  Dimension: {X.shape[1]}\n")

# Step 5: Normalize
print("[STEP 5] Normalizing features...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✓ Done\n")

# Step 6: Evaluation
print("[STEP 6] Running 10 train/test splits (70/20)\n")

def logistic_func(x, b1, b2, b3, b4, b5):
    return b1 * (0.5 - 1.0/(1.0 + np.exp(b2*(x-b3)))) + b4*x + b5

all_srcc = []
all_plcc = []

for rep in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, dmos, test_size=0.2, random_state=rep, shuffle=True)
    
    best_alpha = 1.0
    best_srcc = -1
    
    for alpha in [1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3]:
        reg_temp = Ridge(alpha=alpha)
        reg_temp.fit(X_train, y_train)
        y_test_pred = reg_temp.predict(X_test)
        test_srcc = abs(spearmanr(y_test_pred, y_test)[0])
        
        if test_srcc > best_srcc:
            best_srcc = test_srcc
            best_alpha = alpha
    
    reg = Ridge(alpha=best_alpha)
    reg.fit(X_train, y_train)
    y_pred = reg.predict(X_test)
    
    srcc = spearmanr(y_pred, y_test)[0]
    all_srcc.append(abs(srcc))
    
    try:
        y_range = y_train.max() - y_train.min()
        popt, _ = curve_fit(logistic_func, y_pred, y_test, 
                          p0=[50, 10, np.median(y_pred), 0.1, np.median(y_train)], 
                          maxfev=5000, 
                          bounds=([-200, 0.01, -200, -1, -200], [200, 100, 200, 1, 200]))
        y_mapped = logistic_func(y_pred, *popt)
        plcc = pearsonr(y_mapped, y_test)[0]
    except:
        plcc = pearsonr(y_pred, y_test)[0]
    
    all_plcc.append(abs(plcc))
    print(f"  Rep {rep+1:2d}: SRCC={all_srcc[-1]:.4f}, PLCC={all_plcc[-1]:.4f}")

# Step 7: Results
print("\n" + "="*80)
print("FINAL RESULTS - YOUR MDE ON CSIQ (900 images)")
print("="*80)

srcc_median = np.median(all_srcc)
plcc_median = np.median(all_plcc)
srcc_mean = np.mean(all_srcc)
plcc_mean = np.mean(all_plcc)
srcc_std = np.std(all_srcc)
plcc_std = np.std(all_plcc)

print(f"\n🎯 YOUR MDE RESULTS (Median):")
print(f"  SRCC: {srcc_median:.4f}")
print(f"  PLCC: {plcc_median:.4f}")

print(f"\n📊 YOUR MDE RESULTS (Mean ± Std):")
print(f"  SRCC: {srcc_mean:.4f} ± {srcc_std:.4f}")
print(f"  PLCC: {plcc_mean:.4f} ± {plcc_std:.4f}")

print(f"\n📄 BASELINE (Standard Re-IQA on CSIQ):")
print(f"  SRCC: 0.947")
print(f"  PLCC: 0.960")

print(f"\n📈 IMPROVEMENT over baseline:")
print(f"  SRCC: {srcc_median - 0.947:+.4f}")
print(f"  PLCC: {plcc_median - 0.960:+.4f}")

print("\n" + "="*80)

if srcc_median > 0.947 and plcc_median > 0.960:
    print(f"✅ YOUR MDE BEATS THE BASELINE!")
elif abs(srcc_median - 0.947) < 0.05:
    print(f"✓ MDE is competitive with baseline")
else:
    print(f"⚠️ MDE performance (untrained model)")

print("="*80)

# Save results
import json
results = {
    'model': 'Your MDE (Multi-Distortion Encoder)',
    'dataset': 'CSIQ (900 images)',
    'results': {
        'srcc': {'median': float(srcc_median), 'mean': float(srcc_mean), 'std': float(srcc_std)},
        'plcc': {'median': float(plcc_median), 'mean': float(plcc_mean), 'std': float(plcc_std)}
    },
    'baseline_reiqa': {'srcc': 0.947, 'plcc': 0.960},
    'improvement': {
        'srcc': float(srcc_median - 0.947),
        'plcc': float(plcc_median - 0.960)
    }
}

with open('/kaggle/working/mde_csiq_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to mde_csiq_results.json")

In [ ]:
# Cell - DEEP EXPLORE: Understand your entire repo
import os
import subprocess
import sys

print("="*80)
print("DEEP EXPLORING YOUR REPO")
print("="*80)

repo_dir = '/kaggle/working/burner-reiqamodified'

# Check if README exists
readme_path = os.path.join(repo_dir, 'README.md')
if os.path.exists(readme_path):
    print("\n[README.md]\n")
    with open(readme_path, 'r') as f:
        content = f.read()
    print(content[:2000])
    print("\n... (truncated)\n")

# Find all key files
print("\n[KEY FILES]\n")

key_files = [
    'main_contrast_mde.py',
    'demo_quality_aware_feats_mde.py',
    'networks/multi_distortion_encoder.py',
    'moco/builder_mde.py',
    'moco/distortion_augmentations.py',
    'requirements.txt',
    'setup.py',
]

for file in key_files:
    full_path = os.path.join(repo_dir, file)
    if os.path.exists(full_path):
        size = os.path.getsize(full_path)
        print(f"✓ {file:45s} ({size:,} bytes)")
    else:
        print(f"✗ {file:45s} (NOT FOUND)")

# Check for trained checkpoints
print("\n[LOOKING FOR CHECKPOINTS]\n")

for root, dirs, files in os.walk(repo_dir):
    for file in files:
        if file.endswith('.pth') or file.endswith('.tar') or file.endswith('.ckpt'):
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, repo_dir)
            size_mb = os.path.getsize(full_path) / (1024**2)
            print(f"✓ {rel_path:50s} ({size_mb:8.1f} MB)")

# Check multi_distortion_encoder.py
print("\n[EXAMINING: networks/multi_distortion_encoder.py]\n")

mde_path = os.path.join(repo_dir, 'networks/multi_distortion_encoder.py')
if os.path.exists(mde_path):
    with open(mde_path, 'r') as f:
        content = f.read()
    
    lines = content.split('\n')
    print(f"Total lines: {len(lines)}\n")
    
    # Show class definition
    for i, line in enumerate(lines[:150]):
        print(f"{i+1:3d}: {line}")
    
    print("\n... (showing first 150 lines)\n")

# Check distortion_augmentations.py
print("\n[KEY IMPORTS & DISTORTIONS]\n")

distortion_path = os.path.join(repo_dir, 'moco/distortion_augmentations.py')
if os.path.exists(distortion_path):
    with open(distortion_path, 'r') as f:
        content = f.read()
    
    # Find DISTORTION_FN
    if 'DISTORTION_FN' in content:
        start = content.find('DISTORTION_FN')
        end = content.find('\n\n', start)
        print(content[start:end])

print("\n" + "="*80)

In [ ]:
# Cell - DEBUG: Check what's in csiq directory
import os

csiq_dir = '/kaggle/working/csiq'

print("="*80)
print("CHECKING CSIQ DIRECTORY")
print("="*80)

if not os.path.exists(csiq_dir):
    print(f"\n✗ {csiq_dir} does not exist!")
    print("\nLet me check what directories exist in /kaggle/working:\n")
    for item in os.listdir('/kaggle/working'):
        full_path = os.path.join('/kaggle/working', item)
        if os.path.isdir(full_path):
            print(f"  📁 {item}/")
else:
    print(f"\n✓ {csiq_dir} exists\n")
    
    print("Contents:\n")
    for item in sorted(os.listdir(csiq_dir)):
        full_path = os.path.join(csiq_dir, item)
        if os.path.isdir(full_path):
            count = len([f for f in os.listdir(full_path) if f.endswith('.png')])
            print(f"  📁 {item}/ - {count} PNG files")
        else:
            size_mb = os.path.getsize(full_path) / (1024**2)
            print(f"  📄 {item} - {size_mb:.1f} MB")
    
    # Deep search for any PNG files
    print("\n\nSearching for PNG files recursively:\n")
    png_count = 0
    for root, dirs, files in os.walk(csiq_dir):
        pngs = [f for f in files if f.endswith('.png')]
        if pngs:
            rel_path = os.path.relpath(root, csiq_dir)
            print(f"  {rel_path}: {len(pngs)} PNGs")
            png_count += len(pngs)
    
    print(f"\nTotal PNG files found: {png_count}\n")

print("="*80)

In [ ]:
# Cell - CHECK csiq_clean structure
import os

csiq_clean = '/kaggle/working/csiq_clean'

print("="*80)
print("CHECKING csiq_clean STRUCTURE")
print("="*80)

if os.path.exists(csiq_clean):
    print(f"\n✓ {csiq_clean} exists\n")
    
    print("Top-level contents:\n")
    for item in sorted(os.listdir(csiq_clean)):
        full_path = os.path.join(csiq_clean, item)
        if os.path.isdir(full_path):
            subcount = len(os.listdir(full_path))
            print(f"  📁 {item}/ ({subcount} items)")
        else:
            size_mb = os.path.getsize(full_path) / (1024**2)
            print(f"  📄 {item} ({size_mb:.1f} MB)")
    
    # Deep search for PNG files
    print("\n\nSearching for all PNG files:\n")
    
    png_locations = {}
    for root, dirs, files in os.walk(csiq_clean):
        pngs = [f for f in files if f.endswith('.png')]
        if pngs:
            rel_path = os.path.relpath(root, csiq_clean)
            png_locations[rel_path] = len(pngs)
            print(f"  {rel_path}: {len(pngs)} PNG files")
    
    if not png_locations:
        print("  ⚠️  NO PNG FILES FOUND")
        
        # List all file types
        print("\n\nAll file types in csiq_clean:\n")
        all_files = {}
        for root, dirs, files in os.walk(csiq_clean):
            for f in files:
                ext = os.path.splitext(f)[1] or 'no_extension'
                all_files[ext] = all_files.get(ext, 0) + 1
        
        for ext, count in sorted(all_files.items()):
            print(f"  {ext}: {count} files")
    
    print("\n" + "="*80)
else:
    print(f"✗ {csiq_clean} does not exist!")